# Nutrition5k: experimentos MobileNet

Notebook fino: solo llama a `python -m nutrition5k ...`. Toda la lógica está en `src/nutrition5k/`.

- `outputs/` se enlaza a Google Drive. Cada ejecución guarda `config.json`, `history.csv` (por época),
  `best.pt`, `last.pt`, predicciones y métricas de validación y test.
- Volver a ejecutar una celda es seguro: los runs terminados se saltan y los interrumpidos se reanudan desde `last.pt`.
- Los datos se guardan en Drive como un único `.tar` y se descomprimen en el disco local de Colab, porque leer
  miles de JPEG directamente desde Drive es muy lento.
- Las decisiones (qué backbone o configuración gana) se toman con **validación** en `03_results.ipynb`, nunca con test.
- En RunPod se ejecutan las mismas órdenes `python -m nutrition5k ...` en una terminal (con `tmux`), sin las celdas de Drive.

## Parámetros

In [ ]:
FULL = True          # False: piloto 32/8 para pruebas rápidas
MODE = "complete"    # "complete" (cada 5º frame, todas las cámaras) o "minimal" (1 frame por plato)
EPOCHS = 30
WORKERS = 2          # Colab suele tener 2 vCPU; en RunPod usar más

FULL_FLAG = "--full" if FULL else ""
SPLIT = "full" if FULL else "pilot"

## Preparación del entorno

In [ ]:
from pathlib import Path

REPOSITORY_URL = "https://github.com/victorabellotorres/thesis-multimodal-health-ai.git"
REPOSITORY_DIR = Path("/content/thesis-multimodal-health-ai")
DRIVE_ROOT = Path("/content/drive/MyDrive/nutrition5k")

if (REPOSITORY_DIR / ".git").is_dir():
    !git -C {REPOSITORY_DIR} pull --ff-only
else:
    !git clone {REPOSITORY_URL} {REPOSITORY_DIR}
%cd {REPOSITORY_DIR}
%pip install -q -e .[train]

from google.colab import drive

drive.mount("/content/drive")
# Results live on Drive so nothing is lost when the session ends.
(DRIVE_ROOT / "outputs").mkdir(parents=True, exist_ok=True)
if not Path("outputs").exists():
    Path("outputs").symlink_to(DRIVE_ROOT / "outputs")

Si el `.tar` ya existe en Drive se restaura; si no, `prepare` descarga los vídeos por lotes de 100 platos, extrae los frames reducidos a 320 px y borra los vídeos de cada lote. Después se guarda el `.tar` (solo frames) en Drive para la próxima sesión.

Si `prepare` se interrumpe pero la sesión sigue viva, volver a ejecutar la celda continúa donde se quedó. Si la sesión de Colab se cierra antes de crear el `.tar`, el disco local se pierde y la preparación empieza de nuevo.

In [ ]:
DATA_TAR = DRIVE_ROOT / f"data-{SPLIT}-{MODE}.tar"

if DATA_TAR.is_file():
    !tar -xf {DATA_TAR}
else:
    !python -m nutrition5k fetch
    !python -m nutrition5k prepare --mode {MODE} {FULL_FLAG}
    !tar -cf {DATA_TAR} --exclude='*.h264' data  # frames only, not the videos
!python -m nutrition5k check --mode {MODE} {FULL_FLAG}

## Baseline (media del conjunto de entrenamiento)

In [ ]:
!python -m nutrition5k baseline {FULL_FLAG}

In [ ]:
def train(backbone, unfreeze="all", heads="single", hidden=(512,), image_size=224, seed=0, all_seeds=False):
    """Run one configuration; finished runs are skipped, interrupted ones resume."""
    hidden_args = " ".join(f"--hidden {size}" for size in (hidden or (0,)))
    command = (
        f"python -m nutrition5k train --backbone {backbone} --unfreeze {unfreeze} "
        f"--heads {heads} {hidden_args} --image-size {image_size} --epochs {EPOCHS} "
        f"--num-workers {WORKERS} --seed {seed} --mode {MODE} {FULL_FLAG}"
    )
    if all_seeds:
        command += " --all-seeds"
    print(command)
    get_ipython().system(command)

## Etapa 1: comparación de backbones

Configuración por defecto: fine-tuning completo, cabeza `single`, capa oculta compartida de 512, 224 px.

In [ ]:
for backbone in ("v2", "v3", "v4s", "v4m"):
    train(backbone)

## Etapa 2: ablaciones secuenciales sobre el mejor backbone

Tras cada paso, mirar `03_results.ipynb`, elegir el ganador **por validación** y escribirlo en la variable `BEST_*` del paso siguiente.
Los valores por defecto ya ejecutados en la etapa 1 se reutilizan, no se repiten.

### Paso 1: profundidad del fine-tuning (0 = extracción de características, N = últimas N etapas, all = completo)

In [ ]:
BEST_BACKBONE = "v4s"  # rellenar con el ganador de la etapa 1

for unfreeze in (0, 1, 2):
    train(BEST_BACKBONE, unfreeze=unfreeze)

### Paso 2: agrupación de cabezas (single / grouped = masa, calorías, macros / per-target)

In [ ]:
BEST_UNFREEZE = "all"  # rellenar con el ganador del paso 1

for heads in ("grouped", "per-target"):
    train(BEST_BACKBONE, BEST_UNFREEZE, heads=heads)

### Paso 3: tamaño del tronco compartido

In [ ]:
BEST_HEADS = "single"  # rellenar con el ganador del paso 2

for hidden in ((), (1024, 1024)):
    train(BEST_BACKBONE, BEST_UNFREEZE, BEST_HEADS, hidden=hidden)

### Paso 4: resolución de entrada

In [ ]:
BEST_HIDDEN = (512,)  # rellenar con el ganador del paso 3

train(BEST_BACKBONE, BEST_UNFREEZE, BEST_HEADS, BEST_HIDDEN, image_size=256)

## Etapa 3: semillas

Repetir la configuración final (y las que queden a menos del ruido entre semillas) con las semillas fijas 0, 1 y 2.
La semilla 0 ya está hecha y se reutiliza. `--all-seeds` escribe la media ± SD en `outputs/mobilenet/seeds/<configuración>.json`.

In [ ]:
BEST_IMAGE_SIZE = 224  # rellenar con el ganador del paso 4

train(BEST_BACKBONE, BEST_UNFREEZE, BEST_HEADS, BEST_HIDDEN, BEST_IMAGE_SIZE, all_seeds=True)